In [1]:
import sys
sys.path.insert(0, "..")  # để import từ src/

import cv2
import numpy as np
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

from src.scribble import (
    ScribbleColorizer,
    create_scribble_from_color_image,
    compute_metrics,
    visualize_result,
)
print("Import OK")

Import OK


In [2]:
# Tạo ảnh màu mẫu đơn giản
H, W = 128, 128
color_img = np.zeros((H, W, 3), dtype=np.uint8)
color_img[:64, :64]   = [255, 100,  50]   # góc trái trên: cam
color_img[:64, 64:]   = [ 50, 150, 255]   # góc phải trên: xanh
color_img[64:, :64]   = [ 50, 200, 100]   # góc trái dưới: lục
color_img[64:, 64:]   = [200,  50, 200]   # góc phải dưới: tím

# Tạo grayscale từ ảnh màu (đây là input thực tế)
gray_bgr = cv2.cvtColor(
    cv2.cvtColor(color_img, cv2.COLOR_BGR2GRAY),
    cv2.COLOR_GRAY2BGR
)

# Tự động tạo scribble (lấy 2% pixel làm hint màu)
scribble = create_scribble_from_color_image(color_img, sample_ratio=0.02)

print(f"Color: {color_img.shape}, dtype={color_img.dtype}")
print(f"Gray : {gray_bgr.shape}")
print(f"Scribble pixels: {(scribble.sum(axis=2) > 0).sum()}")

Color: (128, 128, 3), dtype=uint8
Gray : (128, 128, 3)
Scribble pixels: 322


In [ ]:
import mlflow
import yaml

with open("../configs/config.yaml", encoding = "utf-8") as f:
    cfg = yaml.safe_load(f)

# Khởi tạo colorizer với params từ config
colorizer = ScribbleColorizer(
    sigma=cfg["scribble"]["weight_sigma"],
    n_neighbors=cfg["scribble"]["n_neighbors"]
)

# Chạy pipeline
mlflow.set_experiment("scribble-colorization")

with mlflow.start_run(run_name="scribble_test_v1"):
    # Log hyperparams
    mlflow.log_params({
        "sigma": colorizer.sigma,
        "n_neighbors": colorizer.n_neighbors,
        "sample_ratio": 0.02,
        "image_size": f"{H}x{W}",
    })

    # Colorize
    result, info = colorizer.colorize(gray_bgr, scribble)

    # Tính metrics
    metrics = compute_metrics(result, color_img)

    # Log metrics + info
    mlflow.log_metrics({
        "psnr"        : metrics["psnr"],
        "ssim"        : metrics["ssim"],
        "elapsed_sec" : info["elapsed_sec"],
        "n_scribble"  : info["n_scribble_pixels"],
    })

    # Lưu ảnh output
    cv2.imwrite("../results/figures/scribble_result.png", result)
    mlflow.log_artifact("../results/figures/scribble_result.png")

    print(f"PSNR : {metrics['psnr']} dB")
    print(f"SSIM : {metrics['ssim']}")
    print(f"Time : {info['elapsed_sec']}s")
    print(f"Run  : logged to MLflow")

UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 251: character maps to <undefined>